# CLIP 3D Spatial Relationships

## init

In [1]:
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import torch

CLIP_MODEL_NAME = "openai/clip-vit-base-patch16"

## clip model

In [2]:
model = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
model.eval()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e

## data & feature extraction

In [23]:
import json

with open("data/data_min/c_0.9_c_0.1.json", "r") as f:
    json_data = json.load(f)

In [ ]:
image = Image.open("data/apple.jpg")
text = "a red gala apple"

# preprocess
inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True)

# encode
with torch.no_grad():
    outputs = model(**inputs)
    image_emb = outputs.image_embeds
    text_emb = outputs.text_embeds

## attention maps

In [21]:
# model.config.output_attentions = True

with torch.no_grad():
    vision_outputs = model.vision_model(
        pixel_values=inputs["pixel_values"], output_attentions=True
    )
    text_outputs = model.text_model(
        input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], output_attentions=True
    )

# Example: attention from 3rd layer of ViT, head 0
attn_map = vision_outputs.attentions[2][0, 0]  # shape: (num_tokens, num_tokens)
attn_map

tensor([[9.2208e-01, 6.1286e-05, 5.0012e-05,  ..., 2.4218e-04, 2.7706e-04,
         5.0046e-04],
        [4.5923e-02, 7.6754e-02, 6.2790e-02,  ..., 8.8701e-05, 1.3569e-04,
         6.3760e-05],
        [5.2030e-02, 6.1412e-02, 5.0170e-02,  ..., 7.7510e-05, 1.1523e-04,
         5.0666e-05],
        ...,
        [2.0385e-01, 4.8171e-04, 3.4117e-04,  ..., 9.0197e-02, 1.0725e-01,
         2.8560e-02],
        [2.1190e-01, 6.0444e-04, 4.0130e-04,  ..., 9.6132e-02, 1.1625e-01,
         3.6867e-02],
        [2.2307e-01, 5.2531e-04, 3.6921e-04,  ..., 6.7901e-02, 9.9032e-02,
         1.4313e-01]])

In [22]:
vision_outputs.attentions[2].shape

torch.Size([1, 12, 197, 197])

## test

In [32]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch

# Load model and processor
test_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
test_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
test_model.eval()

# Load your image
image = Image.open("vis_10004.jpg")

# Define candidate class labels
labels = ["a photo of a couch", "a photo of a table", "a photo of a pear"]

# Preprocess
inputs = test_processor(text=labels, images=image, return_tensors="pt", padding=True)

# Forward pass
with torch.no_grad():
    outputs = test_model(**inputs)
    logits_per_image = outputs.logits_per_image  # shape: [1, len(labels)]
    probs = logits_per_image.softmax(dim=1)

# Print result
for label, prob in zip(labels, probs[0]):
    print(f"{label}: {prob.item():.4f}")

# Predicted label
pred = labels[probs.argmax()]
print(f"\nPrediction: {pred}")

a photo of a couch: 0.9706
a photo of a table: 0.0294
a photo of a pear: 0.0000

Prediction: a photo of a couch


In [31]:
!echo $HF_HOME

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
